# Phase 3 : Modélisation

Ce notebook charge les données brutes et le pipeline de prétraitement pour entamer la modélisation.

**Étapes :**
1. Chargement de `dataset.csv`
2. Séparation des features `X` et de la cible `y` (`abandoned`)
3. Split stratifié en Train / Validation / Test (70/15/15)
4. Chargement du pipeline `preprocessor.joblib`
5. Vérification de la distribution de la variable cible pour confirmer le déséquilibre.

In [ ]:
import pandas as pd
import numpy as np
import joblib
from sklearn.model_selection import train_test_split

# 1. Chargement des données brutes
df = pd.read_csv('../data/dataset.csv')

# 2. Séparation des features (X) et de la cible (y)
X = df.drop(columns=['abandoned'])
y = df['abandoned']

# 3. Train / Validation / Test split (70/15/15) stratifié
# On sépare d'abord 30% pour un set temporaire (Val + Test)
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.30, stratify=y, random_state=42)

# Puis on divise le set temporaire en deux parties égales (50% Val, 50% Test)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.50, stratify=y_temp, random_state=42)

# 4. Chargement du pipeline de prétraitement
# Assurez-vous d'avoir scikit-learn d'installé dans l'environnement courant
preprocessor = joblib.load('../models/preprocessor.joblib')

# 5. Affichage des shapes et de la distribution
print("Shapes des sous-ensembles :")
print(f"X_train : {X_train.shape}")
print(f"X_val   : {X_val.shape}")
print(f"X_test  : {X_test.shape}")

print("\nDistribution de y_train (%) :")
print((y_train.value_counts(normalize=True) * 100).round(2))

### 2.1 Baseline — Régression Logistique

In [ ]:
from imblearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_validate
import numpy as np

pipeline_lr = Pipeline([
    ('preprocessor', preprocessor),
    ('model', LogisticRegression(class_weight='balanced', max_iter=5000, random_state=42))
])

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scoring = ['f1', 'recall', 'precision']

results_lr = cross_validate(pipeline_lr, X_train, y_train, cv=cv, scoring=scoring, n_jobs=-1)

print("Régression Logistique :")
print(f"F1-score  : {np.mean(results_lr['test_f1']):.3f} ± {np.std(results_lr['test_f1']):.3f}")
print(f"Recall    : {np.mean(results_lr['test_recall']):.3f} ± {np.std(results_lr['test_recall']):.3f}")
print(f"Precision : {np.mean(results_lr['test_precision']):.3f} ± {np.std(results_lr['test_precision']):.3f}")
